## Restructuring (creating organ class folders for frames) in Endoscapes, m2caiSeg, and UDUreter 

In [ ]:
import os
import shutil

# Function to process dataset
def organize_frames_by_masks(base_dir):
    # Define the splits and iterate over them
    splits = ['train', 'val', 'test']
    
    for split in splits:
        masks_dir = os.path.join(base_dir, split, 'Masks')
        frames_dir = os.path.join(base_dir, split, 'Frames')
        
        # Check if Masks and Frames directories exist
        if not os.path.exists(masks_dir):
            print(f"Skipped: Masks directory does not exist for {split} in {base_dir}")
            continue
        if not os.path.exists(frames_dir):
            print(f"Skipped: Frames directory does not exist for {split} in {base_dir}")
            continue
        
        # Walk through each organ class in Masks
        for organ_class in os.listdir(masks_dir):
            organ_class_path = os.path.join(masks_dir, organ_class)
            
            if os.path.isdir(organ_class_path):  # Only process directories
                # Ensure the same folder structure exists in Frames
                organ_class_frames_path = os.path.join(frames_dir, organ_class)
                os.makedirs(organ_class_frames_path, exist_ok=True)
                
                # Process each mask file in the organ class folder
                for mask_file in os.listdir(organ_class_path):
                    mask_file_path = os.path.join(organ_class_path, mask_file)
                    
                    if os.path.isfile(mask_file_path):
                        # Find the corresponding frame file
                        frame_file_name, mask_ext = os.path.splitext(mask_file)
                        frame_file_path = os.path.join(frames_dir, frame_file_name + ".png")
                        
                        # Check for .png first, then .jpg
                        if not os.path.exists(frame_file_path):
                            frame_file_path = os.path.join(frames_dir, frame_file_name + ".jpg")
                        
                        if os.path.exists(frame_file_path):
                            # Copy the frame file to the organ class folder in Frames
                            shutil.copy(frame_file_path, organ_class_frames_path)
                        else:
                            print(f"Frame not found for mask: {mask_file_path}")

# Directories of the datasets
datasets = [
    "...\\Endoscapes",
    "...\\m2caiSeg",
    "...\\UDUreter"
]

# Process each dataset
for dataset_dir in datasets:
    print(f"Processing dataset: {dataset_dir}")
    organize_frames_by_masks(dataset_dir)

print("Organizing frames by masks completed successfully.")


## Restructuring (creating organ class folders for frames) in CholecSeg8k 

In [ ]:
import os
import shutil

def organize_frames_by_masks_with_organ(base_dir):
    # Define the splits and iterate over them
    splits = ['train', 'val', 'test']
    
    for split in splits:
        masks_dir = os.path.join(base_dir, split, 'Masks')
        frames_dir = os.path.join(base_dir, split, 'Frames')
        
        # Check if Masks and Frames directories exist
        if not os.path.exists(masks_dir):
            print(f"Skipped: Masks directory does not exist for {split} in {base_dir}")
            continue
        if not os.path.exists(frames_dir):
            print(f"Skipped: Frames directory does not exist for {split} in {base_dir}")
            continue
        
        # Walk through each organ class in Masks
        for organ_class in os.listdir(masks_dir):
            organ_class_path = os.path.join(masks_dir, organ_class)
            
            if os.path.isdir(organ_class_path):  # Only process directories
                # Process subfolders under each organ class
                for root, _, mask_files in os.walk(organ_class_path):
                    for mask_file in mask_files:
                        mask_file_path = os.path.join(root, mask_file)
                        
                        if os.path.isfile(mask_file_path):
                            # Determine the relative path within the organ class
                            relative_path = os.path.relpath(mask_file_path, organ_class_path)
                            
                            # Find the corresponding frame file path in Frames
                            frame_file_path = os.path.join(frames_dir, relative_path)
                            
                            # Check if the frame file exists
                            if os.path.exists(frame_file_path):
                                # Create the corresponding organ folder in Frames
                                organ_class_frames_path = os.path.join(frames_dir, organ_class)
                                os.makedirs(os.path.join(organ_class_frames_path, os.path.dirname(relative_path)), exist_ok=True)
                                
                                # Copy the frame file to the new organ class folder
                                target_frame_path = os.path.join(organ_class_frames_path, relative_path)
                                shutil.copy(frame_file_path, target_frame_path)
                            else:
                                print(f"Frame not found for mask: {mask_file_path}")

# Directory of the dataset
dataset_dir = "...\\CholecSeg8k"

# Process the dataset
print(f"Processing dataset: {dataset_dir}")
organize_frames_by_masks_with_organ(dataset_dir)

print("Organizing frames by masks with organ folders completed successfully.")


## Confirming the numbers of masks and frames in different classes

In [ ]:
from collections import defaultdict

# Function to count files in each organ class folder
def count_files(base_dir):
    file_counts = defaultdict(lambda: {"Masks_png": 0, "Frames_jpg": 0, "Frames_png": 0})
    
    # Define the splits and iterate over them
    splits = ['train', 'val', 'test']
    
    for split in splits:
        masks_dir = os.path.join(base_dir, split, 'Masks')
        frames_dir = os.path.join(base_dir, split, 'Frames')
        
        if not os.path.exists(masks_dir) or not os.path.exists(frames_dir):
            continue
        
        for organ_class in os.listdir(masks_dir):
            organ_class_path = os.path.join(masks_dir, organ_class)
            
            if os.path.isdir(organ_class_path):  # Only process directories
                # Count PNG files in Masks folder
                mask_png_count = len([f for f in os.listdir(organ_class_path) if f.endswith('.png')])
                file_counts[organ_class]["Masks_png"] += mask_png_count
                
                # Look for corresponding Frames folder
                organ_class_frames_path = os.path.join(frames_dir, organ_class)
                
                if os.path.exists(organ_class_frames_path):
                    # Count JPG and PNG files in Frames folder
                    frame_jpg_count = len([f for f in os.listdir(organ_class_frames_path) if f.endswith('.jpg')])
                    frame_png_count = len([f for f in os.listdir(organ_class_frames_path) if f.endswith('.png')])
                    
                    file_counts[organ_class]["Frames_jpg"] += frame_jpg_count
                    file_counts[organ_class]["Frames_png"] += frame_png_count
    
    return file_counts

# Analyze each dataset
datasets_summary = {}

for dataset_dir in datasets:
    dataset_name = os.path.basename(dataset_dir)
    datasets_summary[dataset_name] = count_files(dataset_dir)

# Display the summary for each dataset
import pandas as pd

for dataset_name, summary in datasets_summary.items():
    df = pd.DataFrame.from_dict(summary, orient="index")
    df.index.name = "Organ Class"
    df.reset_index(inplace=True)
    print(f"\nDataset: {dataset_name}\n")
    display(df)  # Display the dataframe with counts for each organ class and file type


In [ ]:
def count_images_by_class(base_dir):
    splits = ['train', 'val', 'test']
    image_extensions = {'.jpg', '.png'}
    
    for split in splits:
        masks_dir = os.path.join(base_dir, split, 'Masks')
        frames_dir = os.path.join(base_dir, split, 'Frames')
        
        print(f"\nChecking {split} split...")
        
        # Check if directories exist
        if not os.path.exists(masks_dir):
            print(f"Skipped: Masks directory does not exist for {split} in {base_dir}")
            continue
        if not os.path.exists(frames_dir):
            print(f"Skipped: Frames directory does not exist for {split} in {base_dir}")
            continue
        
        # Iterate through organ classes in Masks
        for organ_class in os.listdir(masks_dir):
            organ_masks_path = os.path.join(masks_dir, organ_class)
            organ_frames_path = os.path.join(frames_dir, organ_class)
            
            if not os.path.isdir(organ_masks_path):
                continue  # Skip if not a directory
            
            # Count images in Masks folder for this organ class
            mask_count = sum(
                1 for root, _, files in os.walk(organ_masks_path)
                for file in files if os.path.splitext(file)[1] in image_extensions
            )
            
            # Count images in Frames folder for this organ class
            frame_count = sum(
                1 for root, _, files in os.walk(organ_frames_path)
                for file in files if os.path.splitext(file)[1] in image_extensions
            )
            
            # Print results for this organ class
            print(f"Class: {organ_class}")
            print(f"  Masks: {mask_count} images")
            print(f"  Frames: {frame_count} images")
            
            # Check for discrepancies
            if mask_count != frame_count:
                print(f"  *** Discrepancy found: {mask_count} Masks vs. {frame_count} Frames ***")

# Directory of the dataset
dataset_dir = "...\\CholecSeg8k"

# Run the comparison
count_images_by_class(dataset_dir)


In [ ]:
import os

# Function to delete all .jpg and .png files in the base Frames folders (no subfolder walk)
def delete_files_in_frames(base_dir):
    # Define the splits and iterate over them
    splits = ['train', 'val', 'test']
    
    for split in splits:
        frames_dir = os.path.join(base_dir, split, 'Frames')
        
        # Check if Frames directory exists
        if os.path.exists(frames_dir):
            # List files in the base Frames folder
            for file in os.listdir(frames_dir):
                file_path = os.path.join(frames_dir, file)
                
                # Check if the file is .jpg or .png and delete
                if os.path.isfile(file_path) and (file.endswith('.jpg') or file.endswith('.png')):
                    os.remove(file_path)
                    print(f"Deleted: {file_path}")
        else:
            print(f"Skipped: Frames directory does not exist for {split} in {base_dir}")

# Directories of the datasets
datasets = [
    "...\\Endoscapes",
    "...\\m2caiSeg",
    "...\\UDUreter"
]

# Process each dataset
for dataset_dir in datasets:
    print(f"Processing dataset: {dataset_dir}")
    delete_files_in_frames(dataset_dir)

print("Deletion of .jpg and .png files in Frames folders completed successfully.")



## Folder restructuring (flattening) to store all files in the base folder for 3 datasets

In [ ]:
import os
import shutil

# Function to organize files without nested structure
def flatten_dataset_structure(base_dir):
    splits = ['train', 'val', 'test']
    
    for split in splits:
        annotations_dir = os.path.join(base_dir, split, 'Annotations')
        images_dir = os.path.join(base_dir, split, 'JPEGImages')

        if not os.path.exists(annotations_dir):
            print(f"Skipped: Annotations directory does not exist for {split} in {base_dir}")
            continue
        if not os.path.exists(images_dir):
            print(f"Skipped: JPEGImages directory does not exist for {split} in {base_dir}")
            continue

        # Temporary storage for new file paths to ensure no overwrites
        new_annotations_paths = []
        new_images_paths = []

        # Flatten Annotations directory
        for organ_class in os.listdir(annotations_dir):
            organ_class_path = os.path.join(annotations_dir, organ_class)
            
            if os.path.isdir(organ_class_path):  # Only process directories
                for mask_file in os.listdir(organ_class_path):
                    mask_file_path = os.path.join(organ_class_path, mask_file)
                    
                    if os.path.isfile(mask_file_path):
                        new_name = f"{organ_class}_{mask_file}"
                        new_path = os.path.join(annotations_dir, new_name)
                        new_annotations_paths.append((mask_file_path, new_path))

        # Flatten JPEGImages directory
        for organ_class in os.listdir(images_dir):
            organ_class_path = os.path.join(images_dir, organ_class)
            
            if os.path.isdir(organ_class_path):  # Only process directories
                for frame_file in os.listdir(organ_class_path):
                    frame_file_path = os.path.join(organ_class_path, frame_file)
                    
                    if os.path.isfile(frame_file_path):
                        new_name = f"{organ_class}_{frame_file}"
                        new_path = os.path.join(images_dir, new_name)
                        new_images_paths.append((frame_file_path, new_path))
        
        # Move files to the main folders
        for src, dest in new_annotations_paths:
            shutil.move(src, dest)
        for src, dest in new_images_paths:
            shutil.move(src, dest)

        # Remove empty directories after flattening
        for organ_class in os.listdir(annotations_dir):
            organ_class_path = os.path.join(annotations_dir, organ_class)
            if os.path.isdir(organ_class_path):
                os.rmdir(organ_class_path)
        
        for organ_class in os.listdir(images_dir):
            organ_class_path = os.path.join(images_dir, organ_class)
            if os.path.isdir(organ_class_path):
                os.rmdir(organ_class_path)

# Directories of the datasets
datasets = [
    "...\\Endoscapes",
    "...\\m2caiSeg",
    "...\\UDUreter"
]

# Process each dataset
for dataset_dir in datasets:
    print(f"Processing dataset: {dataset_dir}")
    flatten_dataset_structure(dataset_dir)

print("Dataset restructuring completed successfully.")


## For Dresden

In [ ]:
import os
import shutil

def flatten_dataset_structure_with_subfolders(base_dir):
    splits = ['train', 'val', 'test']
    
    for split in splits:
        annotations_dir = os.path.join(base_dir, split, 'Masks')  # Changed 'Annotations' to 'Masks'
        images_dir = os.path.join(base_dir, split, 'Frames')     # Changed 'JPEGImages' to 'Frames'

        if not os.path.exists(annotations_dir):
            print(f"Skipped: Masks directory does not exist for {split} in {base_dir}")
            continue
        if not os.path.exists(images_dir):
            print(f"Skipped: Frames directory does not exist for {split} in {base_dir}")
            continue

        # Temporary storage for new file paths to ensure no overwrites
        new_annotations_paths = []
        new_images_paths = []

        # Flatten Masks directory
        for organ_class in os.listdir(annotations_dir):
            organ_class_path = os.path.join(annotations_dir, organ_class)
            
            if os.path.isdir(organ_class_path):  # Only process directories
                for root, _, mask_files in os.walk(organ_class_path):
                    subfolder = os.path.relpath(root, organ_class_path).replace("\\", "_").replace("/", "_")
                    for mask_file in mask_files:
                        mask_file_path = os.path.join(root, mask_file)
                        
                        if os.path.isfile(mask_file_path):
                            new_name = f"{organ_class}_{subfolder}_{mask_file}" if subfolder != "." else f"{organ_class}_{mask_file}"
                            new_path = os.path.join(annotations_dir, new_name)
                            new_annotations_paths.append((mask_file_path, new_path))

        # Flatten Frames directory
        for organ_class in os.listdir(images_dir):
            organ_class_path = os.path.join(images_dir, organ_class)
            
            if os.path.isdir(organ_class_path):  # Only process directories
                for root, _, frame_files in os.walk(organ_class_path):
                    subfolder = os.path.relpath(root, organ_class_path).replace("\\", "_").replace("/", "_")
                    for frame_file in frame_files:
                        frame_file_path = os.path.join(root, frame_file)
                        
                        if os.path.isfile(frame_file_path):
                            new_name = f"{organ_class}_{subfolder}_{frame_file}" if subfolder != "." else f"{organ_class}_{frame_file}"
                            new_path = os.path.join(images_dir, new_name)
                            new_images_paths.append((frame_file_path, new_path))
        
        # Move files to the main folders
        for src, dest in new_annotations_paths:
            shutil.move(src, dest)
        for src, dest in new_images_paths:
            shutil.move(src, dest)

        # Remove empty directories after flattening
        for organ_class in os.listdir(annotations_dir):
            organ_class_path = os.path.join(annotations_dir, organ_class)
            if os.path.isdir(organ_class_path):
                shutil.rmtree(organ_class_path)  # Remove the entire tree
        
        for organ_class in os.listdir(images_dir):
            organ_class_path = os.path.join(images_dir, organ_class)
            if os.path.isdir(organ_class_path):
                shutil.rmtree(organ_class_path)  # Remove the entire tree

# Directories of the datasets
datasets = [
    "...\\Dresden"
]

# Process each dataset
for dataset_dir in datasets:
    print(f"Processing dataset: {dataset_dir}")
    flatten_dataset_structure_with_subfolders(dataset_dir)

print("Dataset restructuring completed successfully.")


## For CholecSeg8k

In [ ]:
import os
import shutil

def flatten_dataset_structure_cholecseg(base_dir):
    splits = ['train', 'val', 'test']
    
    for split in splits:
        masks_dir = os.path.join(base_dir, split, 'Masks')  # Path to Masks directory
        frames_dir = os.path.join(base_dir, split, 'Frames')  # Path to Frames directory

        if not os.path.exists(masks_dir):
            print(f"Skipped: Masks directory does not exist for {split} in {base_dir}")
            continue
        if not os.path.exists(frames_dir):
            print(f"Skipped: Frames directory does not exist for {split} in {base_dir}")
            continue

        # Temporary storage for new file paths to ensure no overwrites
        new_masks_paths = []
        new_frames_paths = []

        # Flatten Masks directory
        for organ_class in os.listdir(masks_dir):
            organ_class_path = os.path.join(masks_dir, organ_class)
            
            if os.path.isdir(organ_class_path):  # Only process directories
                for root, _, mask_files in os.walk(organ_class_path):
                    # Determine the relative path within the organ class
                    subfolder = os.path.relpath(root, organ_class_path).replace("\\", "_").replace("/", "_")
                    for mask_file in mask_files:
                        mask_file_path = os.path.join(root, mask_file)
                        
                        if os.path.isfile(mask_file_path):
                            # Construct new name with organ class and subfolder
                            new_name = f"{organ_class}_{subfolder}_{mask_file}" if subfolder != "." else f"{organ_class}_{mask_file}"
                            new_path = os.path.join(masks_dir, new_name)
                            new_masks_paths.append((mask_file_path, new_path))

        # Flatten Frames directory
        for root, _, frame_files in os.walk(frames_dir):
            # Determine the relative path within Frames (use the same relative logic as Masks)
            subfolder = os.path.relpath(root, frames_dir).replace("\\", "_").replace("/", "_")
            for frame_file in frame_files:
                frame_file_path = os.path.join(root, frame_file)
                
                if os.path.isfile(frame_file_path):
                    # Construct new name with subfolder (no organ class in Frames, so we only use subfolder)
                    new_name = f"{subfolder}_{frame_file}" if subfolder != "." else frame_file
                    new_path = os.path.join(frames_dir, new_name)
                    new_frames_paths.append((frame_file_path, new_path))

        # Move files to the main folders
        for src, dest in new_masks_paths:
            shutil.move(src, dest)
        for src, dest in new_frames_paths:
            shutil.move(src, dest)

        # Remove empty directories after flattening
        for organ_class in os.listdir(masks_dir):
            organ_class_path = os.path.join(masks_dir, organ_class)
            if os.path.isdir(organ_class_path):
                shutil.rmtree(organ_class_path)  # Remove the entire tree
        
        for root, dirs, _ in os.walk(frames_dir):
            for dir_name in dirs:
                dir_path = os.path.join(root, dir_name)
                shutil.rmtree(dir_path)  # Remove the entire Frames directory tree

# Directory of the dataset
dataset_dir = "...\\CholecSeg8k"

# Process the dataset
print(f"Processing dataset: {dataset_dir}")
flatten_dataset_structure_cholecseg(dataset_dir)

print("Dataset restructuring completed successfully.")


## Converting all pngs to jpgs

In [ ]:
from PIL import Image
import os

# Directories to process
directories = [
    r"...\CholecSeg8k",
    r"...\Dresden",
    r"...\UDUreter",
    r"...\m2caiSeg",
    r"...\Endoscapes"
]

# Walk through directories and convert PNG files to JPG
def convert_png_to_jpg(base_dir):
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.png'):
                png_path = os.path.join(root, file)
                jpg_path = os.path.splitext(png_path)[0] + '.jpg'
                try:
                    # Open the PNG file
                    with Image.open(png_path) as img:
                        # Save as JPG
                        img.convert("RGB").save(jpg_path, "JPEG")
                    # Remove the original PNG file
                    os.remove(png_path)
                    print(f"Converted and replaced: {png_path} -> {jpg_path}")
                except Exception as e:
                    print(f"Error converting {png_path}: {e}")

# Process each directory
for directory in directories:
    convert_png_to_jpg(directory)


## Replacing names of "Frames" to "JPEGImages" and "Masks" to "Annotations" to be consistent with "Roboflow' SAM2 finetuning workflow

In [ ]:
import os

def replace_folder_names(base_dir):
    splits = ['train', 'val', 'test']
    folder_mapping = {'Frames': 'JPEGImages', 'Masks': 'Annotations'}
    
    for split in splits:
        for old_name, new_name in folder_mapping.items():
            old_path = os.path.join(base_dir, split, old_name)
            new_path = os.path.join(base_dir, split, new_name)
            
            # Rename if the folder exists
            if os.path.exists(old_path):
                os.rename(old_path, new_path)
                print(f"Renamed folder: {old_path} -> {new_path}")
            else:
                print(f"Skipped: {old_path} does not exist")

# List of dataset directories
datasets = [
    "...\\CholecSeg8k",
    "...\\Endoscapes",
    "...\\Dresden",
    "...\\m2caiSeg",
    "...\\UDUreter"
]

# Process each dataset
for dataset_dir in datasets:
    print(f"Processing dataset: {dataset_dir}")
    replace_folder_names(dataset_dir)

print("Folder renaming completed successfully.")


## Converting the jpg masks into JSON files for the "Roboflow" finetuning workflow

In [ ]:
import os
import cv2
import json
import numpy as np
from pycocotools import mask

def convert_each_mask_to_json(dataset_dir):
    """
    Converts each mask in the dataset to a separate JSON file.
    Deletes the original mask files after processing.

    Args:
        dataset_dir (str): Path to the dataset directory containing train/val/test folders.
    """
    # Process train, val, and test splits
    for split in ["train", "val", "test"]:
        image_folder = os.path.join(dataset_dir, split, "JPEGImages")
        annotation_folder = os.path.join(dataset_dir, split, "Annotations")

        if not os.path.exists(image_folder) or not os.path.exists(annotation_folder):
            print(f"Skipped: Missing folder(s) in {split} of {dataset_dir}")
            continue

        # Iterate through all images in JPEGImages
        for image_file in os.listdir(image_folder):
            if not image_file.endswith(".jpg"):
                continue

            # Image metadata
            image_path = os.path.join(image_folder, image_file)
            img = cv2.imread(image_path)
            if img is None:
                print(f"Skipped: Could not read image {image_path}")
                continue

            height, width = img.shape[:2]

            # Process corresponding mask in Annotations
            mask_file = os.path.splitext(image_file)[0] + ".jpg"
            mask_path = os.path.join(annotation_folder, mask_file)

            if os.path.exists(mask_path):
                binary_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if binary_mask is None:
                    print(f"Skipped: Could not read mask {mask_path}")
                    continue

                binary_mask = (binary_mask > 128).astype(np.uint8)  # Convert to binary (0 and 1)

                # Encode segmentation mask using RLE
                rle = mask.encode(np.asfortranarray(binary_mask))
                rle["counts"] = rle["counts"].decode("utf-8")  # RLE counts need to be strings in JSON

                # Calculate area and bbox
                area = float(mask.area(rle))
                bbox = mask.toBbox(rle).tolist()

                # Prepare JSON data
                json_data = {
                    "image": {
                        "image_id": os.path.splitext(image_file)[0],  # Use the image name as ID
                        "license": 1,
                        "file_name": image_file,
                        "height": height,
                        "width": width,
                        "date_captured": "2024-12-07T00:00:00+00:00"
                    },
                    "annotations": [
                        {
                            "id": os.path.splitext(mask_file)[0],  # Use the mask name as ID
                            "bbox": bbox,
                            "area": area,
                            "segmentation": {
                                "counts": rle["counts"],
                                "size": [height, width]
                            }
                        }
                    ]
                }

                # Save JSON file next to the original mask
                json_file_path = os.path.splitext(mask_path)[0] + ".json"
                with open(json_file_path, "w") as json_file:
                    json.dump(json_data, json_file, indent=4)

                # Delete the original mask file
                os.remove(mask_path)
                print(f"Processed and replaced mask: {mask_path} -> {json_file_path}")

# List of dataset directories
datasets = [
    r"...\CholecSeg8k",
    r"...\Endoscapes",
    r"...\Dresden",
    r"...\m2caiSeg",
    #r"...\UDUreter"
]

# Run the conversion for all datasets
for dataset_dir in datasets:
    print(f"Processing dataset: {dataset_dir}")
    convert_each_mask_to_json(dataset_dir)

print("Mask-to-JSON conversion completed for all datasets.")


## Getting the names of all the "Annotations" in the train folder

In [ ]:
# Path to the train folder of Annotations
train_annotations_path = r"...\train\Annotations"

# File to save the JSON file names
output_file_path = r"...\train_annotations_list.txt"

# Check if the directory exists and list all JSON files
if os.path.exists(train_annotations_path):
    json_files = [file for file in os.listdir(train_annotations_path) if file.endswith('.json')]
    
    # Write the file names to a text file
    with open(output_file_path, 'w') as txt_file:
        for file in json_files:
            txt_file.write(file + '\n')
    
    print(f"File names written to: {output_file_path}")
else:
    print(f"The directory {train_annotations_path} does not exist.")


In [17]:
# Check if the directory exists and count all JSON files
if os.path.exists(train_annotations_path):
    json_files_count = len([file for file in os.listdir(train_annotations_path) if file.endswith('.json')])
    print(f"Number of JSON files in the 'train' Annotations folder: {json_files_count}")
else:
    print(f"The directory {train_annotations_path} does not exist.")


Number of JSON files in the 'train' Annotations folder: 50580


## Getting all unique "classes" in the Annotations and JPEGImages of train/val/test and also per class counts

In [ ]:
from collections import defaultdict
import os

# Define the path to the dataset
dataset_path = r"...\AllDatasets"

# Define exceptions where prefixes should include the second part before the underscore for specificity
exceptions = {
    "Cystic": ["Cystic_Artery", "Cystic_Duct", "Cystic_Plate"],
    "Liver": ["Liver_Ligament"]
}

# Handle case-sensitive differentiation for Gall_Bladder
gall_bladder_cases = ["Gall_Bladder", "Gall_bladder"]

# Function to determine the specific prefix
def get_specific_prefix(file, exceptions, gall_bladder_cases):
    file = file.strip()  # Remove any leading/trailing whitespace
    parts = file.split('_')

    # Handle Gall_Bladder cases
    if file.startswith("Gall_Bladder"):
        return "Gall_Bladder"
    if file.startswith("Gall_bladder"):
        return "Gall_bladder"

    # Handle Cystic_Duct cases based on underscores after "Cystic_Duct"
    if file.startswith("Cystic_Duct"):
        remaining_name = file[len("Cystic_Duct"):].strip()  # Extract the part after "Cystic_Duct"
        underscore_count = remaining_name.count('_')
        return f"Cystic_Duct_{underscore_count}_underscores"

    # Handle exceptions for prefixes like Cystic and Liver
    if len(parts) > 1 and parts[0] in exceptions:
        full_prefix = "_".join(parts[:2])
        if full_prefix in exceptions[parts[0]]:
            return full_prefix

    # Handle ureter cases (2 underscores vs 3 underscores)
    if file.startswith("ureter"):
        underscore_count = file.count('_')
        return f"ureter_{underscore_count}_underscores"

    # Handle artery cases (1 underscore vs 3 underscores)
    if file.startswith("artery"):
        underscore_count = file.count('_')
        return f"artery_{underscore_count}_underscores"

    # Handle liver cases (1 underscore vs 2 underscores)
    if file.startswith("liver"):
        underscore_count = file.count('_')
        return f"liver_{underscore_count}_underscores"

    # Default to the first part of the name
    return parts[0]

# Initialize dictionaries for each split
splits = ['train', 'val', 'test']
annotations_class_counts_split = {split: defaultdict(int) for split in splits}
jpegimages_class_counts_split = {split: defaultdict(int) for split in splits}

# Iterate through train, val, and test splits
for split in splits:
    annotations_path = os.path.join(dataset_path, split, 'Annotations')
    jpegimages_path = os.path.join(dataset_path, split, 'JPEGImages')

    # Count for Annotations folder in current split
    if os.path.exists(annotations_path):
        json_files = [file for file in os.listdir(annotations_path) if file.endswith('.json')]
        for file in json_files:
            prefix = get_specific_prefix(file, exceptions, gall_bladder_cases)
            annotations_class_counts_split[split][prefix] += 1

    # Count for JPEGImages folder in current split
    if os.path.exists(jpegimages_path):
        image_files = [file for file in os.listdir(jpegimages_path) if file.endswith(('.jpg', '.jpeg', '.png'))]
        for file in image_files:
            prefix = get_specific_prefix(file, exceptions, gall_bladder_cases)
            jpegimages_class_counts_split[split][prefix] += 1

all_classes = set()
for split in splits:
    all_classes.update(annotations_class_counts_split[split].keys())
    all_classes.update(jpegimages_class_counts_split[split].keys())
all_classes = sorted(all_classes)

# Print Annotations table
print("\nAnnotations Counts:")
print(f"{'Class':<30} {'Train':<10} {'Val':<10} {'Test':<10}")
print("-" * 60)
for class_name in all_classes:
    train_count = annotations_class_counts_split['train'].get(class_name, 0)
    val_count = annotations_class_counts_split['val'].get(class_name, 0)
    test_count = annotations_class_counts_split['test'].get(class_name, 0)
    print(f"{class_name:<30} {train_count:<10} {val_count:<10} {test_count:<10}")

# Print JPEGImages table
print("\nJPEGImages Counts:")
print(f"{'Class':<30} {'Train':<10} {'Val':<10} {'Test':<10}")
print("-" * 60)
for class_name in all_classes:
    train_count = jpegimages_class_counts_split['train'].get(class_name, 0)
    val_count = jpegimages_class_counts_split['val'].get(class_name, 0)
    test_count = jpegimages_class_counts_split['test'].get(class_name, 0)
    print(f"{class_name:<30} {train_count:<10} {val_count:<10} {test_count:<10}")


## Checking if the counts match between JPEGImages vs Annotations

In [ ]:
# Get all unique classes across all splits and sort them alphabetically
all_classes = set()
for split in splits:
    all_classes.update(annotations_class_counts_split[split].keys())
    all_classes.update(jpegimages_class_counts_split[split].keys())
all_classes = sorted(all_classes)

# Function to compare and format cell
def format_cell(annot_count, jpeg_count):
    if annot_count != jpeg_count:
        return f"{annot_count}≠{jpeg_count}!"
    return str(annot_count)

# Print combined table with comparisons
print("\nCounts (Annotations vs JPEGImages):")
print(f"{'Class':<30} {'Train':<15} {'Val':<15} {'Test':<15}")
print("-" * 75)

mismatches_found = False
for class_name in all_classes:
    train_annot = annotations_class_counts_split['train'].get(class_name, 0)
    train_jpeg = jpegimages_class_counts_split['train'].get(class_name, 0)
    val_annot = annotations_class_counts_split['val'].get(class_name, 0)
    val_jpeg = jpegimages_class_counts_split['val'].get(class_name, 0)
    test_annot = annotations_class_counts_split['test'].get(class_name, 0)
    test_jpeg = jpegimages_class_counts_split['test'].get(class_name, 0)
    
    train_cell = format_cell(train_annot, train_jpeg)
    val_cell = format_cell(val_annot, val_jpeg)
    test_cell = format_cell(test_annot, test_jpeg)
    
    if '!' in train_cell + val_cell + test_cell:
        mismatches_found = True
        class_name = f"*{class_name}"  # Mark classes with mismatches
    
    print(f"{class_name:<30} {train_cell:<15} {val_cell:<15} {test_cell:<15}")

if not mismatches_found:
    print("\nAll counts match between Annotations and JPEGImages!")
else:
    print("\nLegend:")
    print("* at start of class name indicates mismatch in one or more splits")
    print("X≠Y! indicates mismatch where X is Annotations count and Y is JPEGImages count")

## Curating the dataset to pick certain classes (31 organ classes) only

In [ ]:
import os
import shutil
from pathlib import Path

# Source and destination paths
src_path = r"...\AllDatasets"
dst_path = r"...\AllDatasets_Curated"

# List of classes to keep
classes_to_keep = {
    "Abdominal", "Cystic_Artery", "Cystic_Duct_2_underscores", "Cystic_Plate",
    "Fat", "Gall_bladder", "Gastrointestinal", "Hepatocystic", "Liver",
    "artery_1_underscores", "colon", "inferior", "intestinal", "nerve",
    "pancreas", "small", "spleen", "stomach", "upperwall",
    "ureter_3_underscores", "vesicular"
}

def create_directory_structure(base_path):
    """Create the directory structure for the curated dataset"""
    for split in ['train', 'val', 'test']:
        for subdir in ['Annotations', 'JPEGImages']:
            Path(os.path.join(base_path, split, subdir)).mkdir(parents=True, exist_ok=True)

def should_keep_file(filename, exceptions, gall_bladder_cases):
    """Check if the file belongs to one of the classes we want to keep"""
    prefix = get_specific_prefix(filename, exceptions, gall_bladder_cases)
    return prefix in classes_to_keep

# Create directory structure
create_directory_structure(dst_path)

# Keep track of copied files
copied_files = {split: {"Annotations": 0, "JPEGImages": 0} for split in splits}

# Copy files for each split
for split in splits:
    # Handle Annotations
    src_annotations = os.path.join(src_path, split, 'Annotations')
    dst_annotations = os.path.join(dst_path, split, 'Annotations')
    
    if os.path.exists(src_annotations):
        for file in os.listdir(src_annotations):
            if file.endswith('.json') and should_keep_file(file, exceptions, gall_bladder_cases):
                shutil.copy2(
                    os.path.join(src_annotations, file),
                    os.path.join(dst_annotations, file)
                )
                copied_files[split]["Annotations"] += 1

    # Handle JPEGImages
    src_images = os.path.join(src_path, split, 'JPEGImages')
    dst_images = os.path.join(dst_path, split, 'JPEGImages')
    
    if os.path.exists(src_images):
        for file in os.listdir(src_images):
            if file.endswith(('.jpg', '.jpeg', '.png')) and should_keep_file(file, exceptions, gall_bladder_cases):
                shutil.copy2(
                    os.path.join(src_images, file),
                    os.path.join(dst_images, file)
                )
                copied_files[split]["JPEGImages"] += 1

# Print summary
print("\nCuration complete! Summary of copied files:")
print("-" * 60)
for split in splits:
    print(f"\n{split.upper()}:")
    print(f"Annotations: {copied_files[split]['Annotations']} files")
    print(f"JPEGImages: {copied_files[split]['JPEGImages']} files")

# Verify the counts match between Annotations and JPEGImages
print("\nVerifying counts match between Annotations and JPEGImages:")
for split in splits:
    annot_count = copied_files[split]['Annotations']
    jpeg_count = copied_files[split]['JPEGImages']
    if annot_count != jpeg_count:
        print(f"WARNING: Mismatch in {split}!")
        print(f"Annotations: {annot_count}, JPEGImages: {jpeg_count}")
    else:
        print(f"{split}: OK ({annot_count} files)")

## Curating the dataset to pick certain classes (21 organ classes) only - 100  (or lesser) random in each class

In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict

# Source and destination paths
src_path = r"...\AllDatasets"
dst_path = r"...\AllDatasets_Curated100"

# List of classes to keep
classes_to_keep = {
    "Abdominal", "Cystic_Artery", "Cystic_Duct_2_underscores", "Cystic_Plate",
    "Fat", "Gall_bladder", "Gastrointestinal", "Hepatocystic", "Liver",
    "artery_1_underscores", "colon", "inferior", "intestinal", "nerve",
    "pancreas", "small", "spleen", "stomach", "upperwall",
    "ureter_3_underscores", "vesicular"
}

# Define exceptions where prefixes should include the second part before the underscore for specificity
exceptions = {
    "Cystic": ["Cystic_Artery", "Cystic_Duct", "Cystic_Plate"],
    "Liver": ["Liver_Ligament"]
}

# Handle case-sensitive differentiation for Gall_Bladder
gall_bladder_cases = ["Gall_Bladder", "Gall_bladder"]

def get_specific_prefix(file, exceptions, gall_bladder_cases):
    """Determine the specific prefix based on file patterns"""
    file = file.strip()
    parts = file.split('_')

    # Handle Gall_Bladder cases
    if file.startswith("Gall_Bladder") or file.startswith("Gall_bladder"):
        return "Gall_bladder"

    # Handle Cystic_Duct cases based on underscores after "Cystic_Duct"
    if file.startswith("Cystic_Duct"):
        remaining_name = file[len("Cystic_Duct"):].strip()
        underscore_count = remaining_name.count('_')
        return f"Cystic_Duct_{underscore_count}_underscores"

    # Handle exceptions for prefixes like Cystic and Liver
    if len(parts) > 1 and parts[0] in exceptions:
        full_prefix = "_".join(parts[:2])
        if full_prefix in exceptions[parts[0]]:
            return full_prefix

    # Handle ureter cases
    if file.startswith("ureter"):
        underscore_count = file.count('_')
        return f"ureter_{underscore_count}_underscores"

    # Handle artery cases
    if file.startswith("artery"):
        underscore_count = file.count('_')
        return f"artery_{underscore_count}_underscores"

    # Default to the first part of the name
    return parts[0]

def create_directory_structure(base_path):
    """Create the directory structure for the curated dataset"""
    for subdir in ['Annotations', 'JPEGImages']:
        Path(os.path.join(base_path, 'train', subdir)).mkdir(parents=True, exist_ok=True)

def collect_files_by_class(src_annotations_path):
    """Group files by their class using the pattern matching logic"""
    files_by_class = defaultdict(list)
    
    for file in os.listdir(src_annotations_path):
        if file.endswith('.json'):
            class_name = get_specific_prefix(file, exceptions, gall_bladder_cases)
            # Only collect files for classes we want to keep
            if class_name in classes_to_keep:
                files_by_class[class_name].append(file)
    
    return files_by_class

def copy_files(src_base, dst_base, filename):
    """Copy both annotation and image files"""
    # Get the image filename (assuming same name, different extension)
    image_filename = os.path.splitext(filename)[0] + '.jpg'
    
    # Copy annotation file
    src_annotation = os.path.join(src_base, 'Annotations', filename)
    dst_annotation = os.path.join(dst_base, 'Annotations', filename)
    shutil.copy2(src_annotation, dst_annotation)
    
    # Copy image file
    src_image = os.path.join(src_base, 'JPEGImages', image_filename)
    dst_image = os.path.join(dst_base, 'JPEGImages', image_filename)
    shutil.copy2(src_image, dst_image)

def main():
    # Create directory structure
    create_directory_structure(dst_path)
    
    # Source paths for train split
    src_train = os.path.join(src_path, 'train')
    src_annotations = os.path.join(src_train, 'Annotations')
    
    # Collect files by class using pattern matching
    files_by_class = collect_files_by_class(src_annotations)
    
    # Statistics for reporting
    class_counts = {}
    total_files = 0
    
    # Process each class that we want to keep
    for class_name in classes_to_keep:
        files = files_by_class[class_name]
        if files:  # Only process if we found files for this class
            # Select up to 100 random files
            selected_files = random.sample(files, min(100, len(files)))
            class_counts[class_name] = len(selected_files)
            total_files += len(selected_files)
            
            # Copy selected files
            for filename in selected_files:
                copy_files(src_train, os.path.join(dst_path, 'train'), filename)
        else:
            class_counts[class_name] = 0
    
    # Print summary
    print("\nCuration complete! Summary of copied files:")
    print("-" * 60)
    for class_name in sorted(classes_to_keep):  # Print all specified classes, even if zero files
        count = class_counts.get(class_name, 0)
        print(f"{class_name}: {count} files")
    print("-" * 60)
    print(f"Total files copied: {total_files}")

if __name__ == "__main__":
    main()

## Curating the dataset to pick certain classes (21 organ classes) only - 50  (or lesser) random in each class

In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict

# Source and destination paths
src_path = r"...\AllDatasets"
dst_path = r"...\AllDatasets_Curated50"

# List of classes to keep
classes_to_keep = {
    "Abdominal", "Cystic_Artery", "Cystic_Duct_2_underscores", "Cystic_Plate",
    "Fat", "Gall_bladder", "Gastrointestinal", "Hepatocystic", "Liver",
    "artery_1_underscores", "colon", "inferior", "intestinal", "nerve",
    "pancreas", "small", "spleen", "stomach", "upperwall",
    "ureter_3_underscores", "vesicular"
}

# Define exceptions where prefixes should include the second part before the underscore for specificity
exceptions = {
    "Cystic": ["Cystic_Artery", "Cystic_Duct", "Cystic_Plate"],
    "Liver": ["Liver_Ligament"]
}

# Handle case-sensitive differentiation for Gall_Bladder
gall_bladder_cases = ["Gall_Bladder", "Gall_bladder"]

def get_specific_prefix(file, exceptions, gall_bladder_cases):
    """Determine the specific prefix based on file patterns"""
    file = file.strip()
    parts = file.split('_')

    # Handle Gall_Bladder cases
    if file.startswith("Gall_Bladder") or file.startswith("Gall_bladder"):
        return "Gall_bladder"

    # Handle Cystic_Duct cases based on underscores after "Cystic_Duct"
    if file.startswith("Cystic_Duct"):
        remaining_name = file[len("Cystic_Duct"):].strip()
        underscore_count = remaining_name.count('_')
        return f"Cystic_Duct_{underscore_count}_underscores"

    # Handle exceptions for prefixes like Cystic and Liver
    if len(parts) > 1 and parts[0] in exceptions:
        full_prefix = "_".join(parts[:2])
        if full_prefix in exceptions[parts[0]]:
            return full_prefix

    # Handle ureter cases
    if file.startswith("ureter"):
        underscore_count = file.count('_')
        return f"ureter_{underscore_count}_underscores"

    # Handle artery cases
    if file.startswith("artery"):
        underscore_count = file.count('_')
        return f"artery_{underscore_count}_underscores"

    # Default to the first part of the name
    return parts[0]

def create_directory_structure(base_path):
    """Create the directory structure for the curated dataset"""
    for subdir in ['Annotations', 'JPEGImages']:
        Path(os.path.join(base_path, 'train', subdir)).mkdir(parents=True, exist_ok=True)

def collect_files_by_class(src_annotations_path):
    """Group files by their class using the pattern matching logic"""
    files_by_class = defaultdict(list)
    
    for file in os.listdir(src_annotations_path):
        if file.endswith('.json'):
            class_name = get_specific_prefix(file, exceptions, gall_bladder_cases)
            # Only collect files for classes we want to keep
            if class_name in classes_to_keep:
                files_by_class[class_name].append(file)
    
    return files_by_class

def copy_files(src_base, dst_base, filename):
    """Copy both annotation and image files"""
    # Get the image filename (assuming same name, different extension)
    image_filename = os.path.splitext(filename)[0] + '.jpg'
    
    # Copy annotation file
    src_annotation = os.path.join(src_base, 'Annotations', filename)
    dst_annotation = os.path.join(dst_base, 'Annotations', filename)
    shutil.copy2(src_annotation, dst_annotation)
    
    # Copy image file
    src_image = os.path.join(src_base, 'JPEGImages', image_filename)
    dst_image = os.path.join(dst_base, 'JPEGImages', image_filename)
    shutil.copy2(src_image, dst_image)

def main():
    # Create directory structure
    create_directory_structure(dst_path)
    
    # Source paths for train split
    src_train = os.path.join(src_path, 'train')
    src_annotations = os.path.join(src_train, 'Annotations')
    
    # Collect files by class using pattern matching
    files_by_class = collect_files_by_class(src_annotations)
    
    # Statistics for reporting
    class_counts = {}
    total_files = 0
    
    # Process each class that we want to keep
    for class_name in classes_to_keep:
        files = files_by_class[class_name]
        if files:  # Only process if we found files for this class
            # Select up to 100 random files
            selected_files = random.sample(files, min(50, len(files)))
            class_counts[class_name] = len(selected_files)
            total_files += len(selected_files)
            
            # Copy selected files
            for filename in selected_files:
                copy_files(src_train, os.path.join(dst_path, 'train'), filename)
        else:
            class_counts[class_name] = 0
    
    # Print summary
    print("\nCuration complete! Summary of copied files:")
    print("-" * 60)
    for class_name in sorted(classes_to_keep):  # Print all specified classes, even if zero files
        count = class_counts.get(class_name, 0)
        print(f"{class_name}: {count} files")
    print("-" * 60)
    print(f"Total files copied: {total_files}")

if __name__ == "__main__":
    main()

## Curating the dataset to pick certain classes (21 organ classes) only - 200 (or lesser) random in each class

In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict

# Source and destination paths
src_path = r"...\AllDatasets"
dst_path = r"...\AllDatasets_Curated200"

# List of classes to keep
classes_to_keep = {
    "Abdominal", "Cystic_Artery", "Cystic_Duct_2_underscores", "Cystic_Plate",
    "Fat", "Gall_bladder", "Gastrointestinal", "Hepatocystic", "Liver",
    "artery_1_underscores", "colon", "inferior", "intestinal", "nerve",
    "pancreas", "small", "spleen", "stomach", "upperwall",
    "ureter_3_underscores", "vesicular"
}

# Define exceptions where prefixes should include the second part before the underscore for specificity
exceptions = {
    "Cystic": ["Cystic_Artery", "Cystic_Duct", "Cystic_Plate"],
    "Liver": ["Liver_Ligament"]
}

# Handle case-sensitive differentiation for Gall_Bladder
gall_bladder_cases = ["Gall_Bladder", "Gall_bladder"]

def get_specific_prefix(file, exceptions, gall_bladder_cases):
    """Determine the specific prefix based on file patterns"""
    file = file.strip()
    parts = file.split('_')

    # Handle Gall_Bladder cases
    if file.startswith("Gall_Bladder") or file.startswith("Gall_bladder"):
        return "Gall_bladder"

    # Handle Cystic_Duct cases based on underscores after "Cystic_Duct"
    if file.startswith("Cystic_Duct"):
        remaining_name = file[len("Cystic_Duct"):].strip()
        underscore_count = remaining_name.count('_')
        return f"Cystic_Duct_{underscore_count}_underscores"

    # Handle exceptions for prefixes like Cystic and Liver
    if len(parts) > 1 and parts[0] in exceptions:
        full_prefix = "_".join(parts[:2])
        if full_prefix in exceptions[parts[0]]:
            return full_prefix

    # Handle ureter cases
    if file.startswith("ureter"):
        underscore_count = file.count('_')
        return f"ureter_{underscore_count}_underscores"

    # Handle artery cases
    if file.startswith("artery"):
        underscore_count = file.count('_')
        return f"artery_{underscore_count}_underscores"

    # Default to the first part of the name
    return parts[0]

def create_directory_structure(base_path):
    """Create the directory structure for the curated dataset"""
    for subdir in ['Annotations', 'JPEGImages']:
        Path(os.path.join(base_path, 'train', subdir)).mkdir(parents=True, exist_ok=True)

def collect_files_by_class(src_annotations_path):
    """Group files by their class using the pattern matching logic"""
    files_by_class = defaultdict(list)
    
    for file in os.listdir(src_annotations_path):
        if file.endswith('.json'):
            class_name = get_specific_prefix(file, exceptions, gall_bladder_cases)
            # Only collect files for classes we want to keep
            if class_name in classes_to_keep:
                files_by_class[class_name].append(file)
    
    return files_by_class

def copy_files(src_base, dst_base, filename):
    """Copy both annotation and image files"""
    # Get the image filename (assuming same name, different extension)
    image_filename = os.path.splitext(filename)[0] + '.jpg'
    
    # Copy annotation file
    src_annotation = os.path.join(src_base, 'Annotations', filename)
    dst_annotation = os.path.join(dst_base, 'Annotations', filename)
    shutil.copy2(src_annotation, dst_annotation)
    
    # Copy image file
    src_image = os.path.join(src_base, 'JPEGImages', image_filename)
    dst_image = os.path.join(dst_base, 'JPEGImages', image_filename)
    shutil.copy2(src_image, dst_image)

def main():
    # Create directory structure
    create_directory_structure(dst_path)
    
    # Source paths for train split
    src_train = os.path.join(src_path, 'train')
    src_annotations = os.path.join(src_train, 'Annotations')
    
    # Collect files by class using pattern matching
    files_by_class = collect_files_by_class(src_annotations)
    
    # Statistics for reporting
    class_counts = {}
    total_files = 0
    
    # Process each class that we want to keep
    for class_name in classes_to_keep:
        files = files_by_class[class_name]
        if files:  # Only process if we found files for this class
            # Select up to 100 random files
            selected_files = random.sample(files, min(200, len(files)))
            class_counts[class_name] = len(selected_files)
            total_files += len(selected_files)
            
            # Copy selected files
            for filename in selected_files:
                copy_files(src_train, os.path.join(dst_path, 'train'), filename)
        else:
            class_counts[class_name] = 0
    
    # Print summary
    print("\nCuration complete! Summary of copied files:")
    print("-" * 60)
    for class_name in sorted(classes_to_keep):  # Print all specified classes, even if zero files
        count = class_counts.get(class_name, 0)
        print(f"{class_name}: {count} files")
    print("-" * 60)
    print(f"Total files copied: {total_files}")

if __name__ == "__main__":
    main()

## Curating the dataset to pick certain classes (21 organ classes) only - 400 random (or lesser) in each class

In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict

# Source and destination paths
src_path = r"...\AllDatasets"
dst_path = r"...\AllDatasets_Curated400"

# List of classes to keep
classes_to_keep = {
    "Abdominal", "Cystic_Artery", "Cystic_Duct_2_underscores", "Cystic_Plate",
    "Fat", "Gall_bladder", "Gastrointestinal", "Hepatocystic", "Liver",
    "artery_1_underscores", "colon", "inferior", "intestinal", "nerve",
    "pancreas", "small", "spleen", "stomach", "upperwall",
    "ureter_3_underscores", "vesicular"
}

# Define exceptions where prefixes should include the second part before the underscore for specificity
exceptions = {
    "Cystic": ["Cystic_Artery", "Cystic_Duct", "Cystic_Plate"],
    "Liver": ["Liver_Ligament"]
}

# Handle case-sensitive differentiation for Gall_Bladder
gall_bladder_cases = ["Gall_Bladder", "Gall_bladder"]

def get_specific_prefix(file, exceptions, gall_bladder_cases):
    """Determine the specific prefix based on file patterns"""
    file = file.strip()
    parts = file.split('_')

    # Handle Gall_Bladder cases
    if file.startswith("Gall_Bladder") or file.startswith("Gall_bladder"):
        return "Gall_bladder"

    # Handle Cystic_Duct cases based on underscores after "Cystic_Duct"
    if file.startswith("Cystic_Duct"):
        remaining_name = file[len("Cystic_Duct"):].strip()
        underscore_count = remaining_name.count('_')
        return f"Cystic_Duct_{underscore_count}_underscores"

    # Handle exceptions for prefixes like Cystic and Liver
    if len(parts) > 1 and parts[0] in exceptions:
        full_prefix = "_".join(parts[:2])
        if full_prefix in exceptions[parts[0]]:
            return full_prefix

    # Handle ureter cases
    if file.startswith("ureter"):
        underscore_count = file.count('_')
        return f"ureter_{underscore_count}_underscores"

    # Handle artery cases
    if file.startswith("artery"):
        underscore_count = file.count('_')
        return f"artery_{underscore_count}_underscores"

    # Default to the first part of the name
    return parts[0]

def create_directory_structure(base_path):
    """Create the directory structure for the curated dataset"""
    for subdir in ['Annotations', 'JPEGImages']:
        Path(os.path.join(base_path, 'train', subdir)).mkdir(parents=True, exist_ok=True)

def collect_files_by_class(src_annotations_path):
    """Group files by their class using the pattern matching logic"""
    files_by_class = defaultdict(list)
    
    for file in os.listdir(src_annotations_path):
        if file.endswith('.json'):
            class_name = get_specific_prefix(file, exceptions, gall_bladder_cases)
            # Only collect files for classes we want to keep
            if class_name in classes_to_keep:
                files_by_class[class_name].append(file)
    
    return files_by_class

def copy_files(src_base, dst_base, filename):
    """Copy both annotation and image files"""
    # Get the image filename (assuming same name, different extension)
    image_filename = os.path.splitext(filename)[0] + '.jpg'
    
    # Copy annotation file
    src_annotation = os.path.join(src_base, 'Annotations', filename)
    dst_annotation = os.path.join(dst_base, 'Annotations', filename)
    shutil.copy2(src_annotation, dst_annotation)
    
    # Copy image file
    src_image = os.path.join(src_base, 'JPEGImages', image_filename)
    dst_image = os.path.join(dst_base, 'JPEGImages', image_filename)
    shutil.copy2(src_image, dst_image)

def main():
    # Create directory structure
    create_directory_structure(dst_path)
    
    # Source paths for train split
    src_train = os.path.join(src_path, 'train')
    src_annotations = os.path.join(src_train, 'Annotations')
    
    # Collect files by class using pattern matching
    files_by_class = collect_files_by_class(src_annotations)
    
    # Statistics for reporting
    class_counts = {}
    total_files = 0
    
    # Process each class that we want to keep
    for class_name in classes_to_keep:
        files = files_by_class[class_name]
        if files:  # Only process if we found files for this class
            # Select up to 100 random files
            selected_files = random.sample(files, min(400, len(files)))
            class_counts[class_name] = len(selected_files)
            total_files += len(selected_files)
            
            # Copy selected files
            for filename in selected_files:
                copy_files(src_train, os.path.join(dst_path, 'train'), filename)
        else:
            class_counts[class_name] = 0
    
    # Print summary
    print("\nCuration complete! Summary of copied files:")
    print("-" * 60)
    for class_name in sorted(classes_to_keep):  # Print all specified classes, even if zero files
        count = class_counts.get(class_name, 0)
        print(f"{class_name}: {count} files")
    print("-" * 60)
    print(f"Total files copied: {total_files}")

if __name__ == "__main__":
    main()

## Names of all annotation files in a txt

In [ ]:
import os

# Define the directory path
directory_path = r'...\AllDatasets\train\Annotations'

# Get the list of all file names in the directory
file_names = os.listdir(directory_path)

# Define the path for the output text file
output_file_path = r'...\AllDatasets\Annotations_FileNames.txt'

# Write the file names into a text file
with open(output_file_path, 'w') as f:
    for file_name in file_names:
        f.write(file_name + '\n')

output_file_path


In [ ]:
import os

# Define the directory path
directory_path = r'...\AllDatasets_Curated100\train\Annotations'

# Get the list of all file names in the directory
file_names = os.listdir(directory_path)

# Define the path for the output text file
output_file_path = r'...\AllDatasets_Curated100\Annotations_FileNames.txt'

# Write the file names into a text file
with open(output_file_path, 'w') as f:
    for file_name in file_names:
        f.write(file_name + '\n')

output_file_path

In [ ]:
import os

# Define the directory path
directory_path = r'...\AllDatasets_Curated50\train\Annotations'

# Get the list of all file names in the directory
file_names = os.listdir(directory_path)

# Define the path for the output text file
output_file_path = r'...\AllDatasets_Curated50\Annotations_FileNames.txt'

# Write the file names into a text file
with open(output_file_path, 'w') as f:
    for file_name in file_names:
        f.write(file_name + '\n')

output_file_path

In [ ]:
import os

# Define the directory path
directory_path = r'...\AllDatasets_Curated200\train\Annotations'

# Get the list of all file names in the directory
file_names = os.listdir(directory_path)

# Define the path for the output text file
output_file_path = r'...\AllDatasets_Curated200\Annotations_FileNames.txt'

# Write the file names into a text file
with open(output_file_path, 'w') as f:
    for file_name in file_names:
        f.write(file_name + '\n')

output_file_path

In [ ]:
import os

# Define the directory path
directory_path = r'...\AllDatasets_Curated400\train\Annotations'

# Get the list of all file names in the directory
file_names = os.listdir(directory_path)

# Define the path for the output text file
output_file_path = r'...\AllDatasets_Curated400\Annotations_FileNames.txt'

# Write the file names into a text file
with open(output_file_path, 'w') as f:
    for file_name in file_names:
        f.write(file_name + '\n')

output_file_path